# Claim Document AI: zero-shot vs. OCR + LLM vs. fine-tuned Qwen2-VL (Colab T4)

**Runtime → T4 GPU → Run all.** About 3 hours end to end (set `LIMIT` / `EPOCHS` lower for a quick pass).

Produces the comparison table for the README / portfolio, on the official test splits:
- **CORD-v2** receipts (100 test receipts) and **FUNSD** forms (50 test forms)
- systems: Qwen2-VL-2B zero-shot · Tesseract OCR + LLM (the usual pipeline) · Qwen2-VL-2B fine-tuned with QLoRA
- metric: field-level F1 (+ strict F1, perfect documents, JSON failures, seconds/doc), and the share of receipts whose arithmetic checks pass

In [ ]:
!nvidia-smi -L
!apt-get -qq install -y tesseract-ocr > /dev/null
!git clone -q https://github.com/alihassan891u-lab/claim-document-ai.git || true
%cd claim-document-ai
!pip -q install -e ".[gpu,ocr,llm,dev]"
LIMIT, EPOCHS = None, 2          # e.g. LIMIT=30, EPOCHS=1 for a quick run
BASELINE_LLM = "hf"              # "hf" = local Qwen2.5-1.5B (free); "claude" / "openai" need an API key in the env

In [ ]:
# FUNSD (CORD-v2 is streamed from the Hugging Face Hub)
!mkdir -p data/funsd && wget -q https://guillaumejaume.github.io/FUNSD/dataset.zip -O /tmp/funsd.zip && unzip -q -o /tmp/funsd.zip -d data/funsd
from claimdoc.datasets import cord_from_hf, funsd_from_dir
cord_test = list(cord_from_hf("test"))[:LIMIT]
funsd_test = list(funsd_from_dir("data/funsd", "testing_data"))[:LIMIT]
print(len(cord_test), "receipts,", len(funsd_test), "forms"); print(cord_test[0].target)

## 1. Zero-shot Qwen2-VL-2B

In [ ]:
import json, gc, torch
from claimdoc.evaluate import run_system, table
from claimdoc.vlm import QwenVLExtractor
results = {"CORD receipts (test)": [], "FUNSD forms (test)": []}
def run(ex, tag):
    results["CORD receipts (test)"].append(run_system(ex, cord_test, f"results/pred_{tag}_cord.jsonl"))
    results["FUNSD forms (test)"].append(run_system(ex, funsd_test, f"results/pred_{tag}_funsd.jsonl"))
    print({k: v[-1]["loose"]["f1"] for k, v in results.items()})
ex = QwenVLExtractor("Qwen/Qwen2-VL-2B-Instruct"); run(ex, "zeroshot"); del ex; gc.collect(); torch.cuda.empty_cache()

## 2. Baseline: Tesseract OCR + LLM

In [ ]:
from claimdoc.baseline import OcrLlmExtractor, hf_llm, claude_llm, openai_llm
llm = {"hf": hf_llm, "claude": claude_llm, "openai": openai_llm}[BASELINE_LLM]()
ex = OcrLlmExtractor(llm, name=f"Tesseract + {BASELINE_LLM} LLM"); run(ex, "ocrllm"); del ex, llm; gc.collect(); torch.cuda.empty_cache()

## 3. Fine-tune Qwen2-VL-2B with QLoRA (CORD train 800 + FUNSD train 149)

In [ ]:
from claimdoc.finetune import train
train_ex = list(cord_from_hf("train")) + list(funsd_from_dir("data/funsd", "training_data"))
adapter = train(train_ex, "outputs/qwen2vl-claims-lora", epochs=EPOCHS)
gc.collect(); torch.cuda.empty_cache()

## 4. Fine-tuned model on the same test sets

In [ ]:
ex = QwenVLExtractor("Qwen/Qwen2-VL-2B-Instruct", adapter="outputs/qwen2vl-claims-lora"); run(ex, "lora")

## 5. Comparison table (send `results.zip` back)

In [ ]:
md = "\n".join(table(v, k) for k, v in results.items())
for k, v in results.items():
    for r in v: md += f"\n{k} · {r['system']}: receipts auto-ready {r['receipts_auto_ready']}, JSON {r['json']}"
open("results/comparison.md", "w").write(md); json.dump(results, open("results/comparison.json", "w"), indent=2, ensure_ascii=False)
print(md)
!zip -qr results.zip results
!cd outputs && zip -qr ../adapter.zip qwen2vl-claims-lora
from google.colab import files; files.download("results.zip")